# Build a cited HDB research brief

Run from the project folder with the Python environment described in the README. These cells use the bundled transaction snapshot and the offline graph. They demonstrate the flow through Perceive, Reason, Action and Learn. Live Gemini and OneMap use the terminal instructions in the README.


In [ ]:
from pathlib import Path
from uuid import uuid4

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command
from hdb_advisor.agents import offline_intake
from hdb_advisor.graph import RECURSION_LIMIT, Settings, build_graph, prepare_turn
from hdb_advisor.market import provenance, search_market
from hdb_advisor.policy import retrieve_rules, evaluate_grants
from hdb_advisor.state import BuyerProfile

database = Path("data/hdb.db")
assert database.is_file(), "Open this notebook from the project folder with the bundled database."
source = provenance(database)
print(source.rows, "transactions through", source.last_month)


## Perceive the buyer brief

The parser translates a sentence into a typed record. Inspect the output and look for fields that were left unknown. During a Gemini run the Intake agent fills the same Pydantic contract.


In [ ]:
brief = "4-room under S$600k, last 12 months, weekday 8am to NTU, with grants"
profile = offline_intake(brief)
profile.model_dump(exclude_unset=True)


## Check a tool without the graph

A prepared SQLite query filters the snapshot and Python computes the per-town quartiles. Change the budget in the profile, run this cell again, and compare the town counts. Then inspect the source query in `hdb_advisor/market.py`.


In [ ]:
market = search_market(database, profile)
for town in market.towns[:3]:
    print(f"{town.town} | {town.n_tx} transactions | median S${town.median:,.0f}")


## Retrieve evidence and handle unknowns

Each policy passage carries its source, section and snapshot date. The grant tool uses a separate reviewed rule table. Income is missing in this example, so the Family Grant and EHG screening ask for it.


In [ ]:
for passage in retrieve_rules("enhanced grant income and employment"):
    print(passage.title, "|", passage.section, "|", passage.snapshot_date)

grant_profile = BuyerProfile(flat_type="4 ROOM", citizenship="SC_SC", first_timer=True)
for grant in evaluate_grants(grant_profile):
    print(grant.name, grant.status, grant.missing_fields)


## Reason and pause before action

The graph saves state at the plan gate. In this notebook, the next cell acts as the human and makes one edit. Try cancelling in a fresh session and observe whether the market query ran.


In [ ]:
memory = InMemoryStore()
app = build_graph(Settings(database), InMemorySaver(), memory)
config = {"configurable": {"thread_id": uuid4().hex, "user_id": "walkthrough"},
          "recursion_limit": RECURSION_LIMIT}
pending = app.invoke(prepare_turn(brief), config)
gate = pending["__interrupt__"][0].value
assert gate["type"] == "confirm_plan"
gate


In [ ]:
response = app.invoke(Command(resume={
    "action": "edit", "changes": {"town": "JURONG WEST"}, "remember": True,
}), config)
print(response["canonical_report"])


## Learn from the approved correction

The graph keeps a reusable town preference and a corrected example for this local user. In production these use separate SQLite storage. Start a second thread with the same user and check which filter carries over.


In [ ]:
new_config = {"configurable": {"thread_id": uuid4().hex, "user_id": "walkthrough"},
              "recursion_limit": RECURSION_LIMIT}
new_turn = app.invoke(prepare_turn("4-room under 650k"), new_config)
new_filters = new_turn["__interrupt__"][0].value["filters"]
assert new_filters["town"] == "JURONG WEST"
print(new_filters)
print(memory.search(("examples", "walkthrough", "intake"))[0].value)


## Verify the behavior

Run `python -m pytest -q` for the automated checks and `python -m evals.run_evaluation` for the baseline comparison. Open `evals/test_workflow.py` and trace an individual case from the labelled input to its assertion. Try adding a new wording of the same buyer request and check which route it takes.
